# Predictive Analytics - Marketing Sales Forecast

**Edutech Solution Data Analytics Internship - Task 12**

This notebook builds a Linear Regression model using a marketing dataset. The objective is to forecast future sales outcomes from advertising spend and discount percentage.

## Assignment Requirements

- Tool: Python and Scikit-learn
- Dataset: Marketing dataset
- Method: Linear Regression
- Tasks: Model evaluation and future predictions
- Deliverables: Prediction report and saved model file

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

ROOT = Path.cwd()
if not (ROOT / "data" / "marketing_dataset.csv").exists():
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data" / "marketing_dataset.csv"
MODEL_PATH = ROOT / "models" / "linear_regression_model.pkl"
REPORTS_DIR = ROOT / "reports"

FEATURE_COLUMNS = [
    "tv_ad_spend",
    "radio_ad_spend",
    "social_media_spend",
    "email_campaign_spend",
    "discount_percent",
]
TARGET_COLUMN = "sales"

## Load Dataset

The dataset contains campaign-level marketing spend and the sales outcome for each campaign.

In [ ]:
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

## Prepare Train and Test Data

The target variable is `sales`. The model uses the marketing spend columns and discount percentage as input features.

In [ ]:
X = df[FEATURE_COLUMNS]
y = df[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training rows: {len(X_train)}")
print(f"Testing rows: {len(X_test)}")

## Train Linear Regression Model

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

print("Intercept:", round(model.intercept_, 2))
pd.DataFrame({"feature": FEATURE_COLUMNS, "coefficient": model.coef_})

## Evaluate Model

RMSE and MAE measure prediction error. R2 score measures how much variation in sales is explained by the model.

In [ ]:
y_pred = model.predict(X_test)

rmse = mean_squared_error(y_test, y_pred) ** 0.5
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

metrics = pd.DataFrame(
    {
        "metric": ["RMSE", "MAE", "R2 Score"],
        "value": [round(rmse, 2), round(mae, 2), round(r2, 4)],
    }
)
metrics

In [ ]:
prediction_results = X_test.copy()
prediction_results["actual_sales"] = y_test.values
prediction_results["predicted_sales"] = y_pred.round(2)
prediction_results["prediction_error"] = (
    prediction_results["actual_sales"] - prediction_results["predicted_sales"]
).round(2)

prediction_results.head()

## Actual vs Predicted Sales

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.75)
lower = min(y_test.min(), y_pred.min())
upper = max(y_test.max(), y_pred.max())
plt.plot([lower, upper], [lower, upper], color="red")
plt.title("Actual vs Predicted Sales")
plt.xlabel("Actual Sales")
plt.ylabel("Predicted Sales")
plt.show()

A generated chart is also saved in the project at `reports/actual_vs_predicted.png`.

![Actual vs Predicted Sales](../reports/actual_vs_predicted.png)

## Make Future Predictions

In [ ]:
future_campaigns = pd.DataFrame(
    [
        {
            "campaign_name": "Balanced Awareness Campaign",
            "tv_ad_spend": 45000,
            "radio_ad_spend": 12000,
            "social_media_spend": 22000,
            "email_campaign_spend": 6000,
            "discount_percent": 10,
        },
        {
            "campaign_name": "Digital Growth Campaign",
            "tv_ad_spend": 25000,
            "radio_ad_spend": 8000,
            "social_media_spend": 38000,
            "email_campaign_spend": 9000,
            "discount_percent": 15,
        },
        {
            "campaign_name": "Premium Launch Campaign",
            "tv_ad_spend": 70000,
            "radio_ad_spend": 20000,
            "social_media_spend": 35000,
            "email_campaign_spend": 11000,
            "discount_percent": 5,
        },
    ]
)

future_campaigns["predicted_sales"] = model.predict(
    future_campaigns[FEATURE_COLUMNS]
).round(2)
future_campaigns

## Save Model

The trained model is saved as `models/linear_regression_model.pkl` so it can be reused for future predictions.

In [ ]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(
    {
        "model": model,
        "feature_columns": FEATURE_COLUMNS,
        "target_column": TARGET_COLUMN,
        "metrics": {
            "rmse": round(rmse, 2),
            "mae": round(mae, 2),
            "r2_score": round(r2, 4),
        },
    },
    MODEL_PATH,
)
print(f"Model saved to: {MODEL_PATH}")

## Conclusion

The Linear Regression model forecasts future sales outcomes using marketing campaign inputs. The project includes the notebook, dataset, saved model file, prediction report, future predictions, and visualization required for submission.

**Interview answer - Regression vs Classification:** Regression predicts continuous values such as sales. Classification predicts categories such as yes/no or high/medium/low.

**Interview answer - RMSE:** RMSE means Root Mean Squared Error. It shows the average prediction error in the same unit as the target variable.